# Makata: verified Tagalog comprehension baselines on free Colab

Use **Runtime → Change runtime type → T4 GPU** (or another available free NVIDIA GPU).
Use only the free tier. No AWS, API credits, paid subscriptions, Drive mount, or public uploads are used.
GPU allocation and session lifetime are not guaranteed. Run cells in order and remain present.

This notebook embeds the current research source, including unpushed fixes. It preserves the exact
FilBench/model/tokenizer/dataset pins, zero-shot prompts, **float32** scoring, batch one and no truncation.
CUDA replaces CPU explicitly; TF32 and quantization are disabled. It freezes a separate GPU execution
profile before candidate weights or benchmark examples load. All results remain private to this runtime
until **you download them**. A terminated runtime can lose undownloaded artifacts; incomplete runs do not count.

First validate the environment and both candidates on 20 **synthetic development questions**.
Only the later baseline cells access the full 900-question held-out dataset. Do not tune against those results.


## 1. Verify and unpack the reviewed source; check GPU assignment

In [ ]:
import base64, hashlib, io, json, os, pathlib, subprocess, sys, uuid, zipfile

# This allowlisted snapshot includes unpushed fixes, not your entire repository.
PAYLOAD = '...'
BUNDLE_SHA256 = 'ac9949705cd1f5f5e6e9e266afb8b6f840c46ae7fab584ba6d8f34d236e05fde'
data = base64.b64decode(PAYLOAD)
assert hashlib.sha256(data).hexdigest() == BUNDLE_SHA256
ROOT = pathlib.Path('/content') / ('makata-' + BUNDLE_SHA256[:12])
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    manifest = json.loads(archive.read('bundle-manifest.json'))
    assert set(archive.namelist()) == set(manifest['files_sha256']) | {'bundle-manifest.json'}
    for name, expected in manifest['files_sha256'].items():
        target = (ROOT / name).resolve()
        assert target.is_relative_to(ROOT.resolve())
        content = archive.read(name)
        assert hashlib.sha256(content).hexdigest() == expected
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert target.read_bytes() == content, 'Existing source differs; start a fresh runtime.'
        else:
            target.write_bytes(content)
(ROOT / 'bundle-manifest.json').write_text(json.dumps(manifest, indent=2))
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
SESSION = ROOT / 'research/runs' / ('colab-' + uuid.uuid4().hex[:12])
SESSION.mkdir(parents=True)
subprocess.run(['nvidia-smi'], check=True)
print('Source verified:', BUNDLE_SHA256)
print('Result session:', SESSION)


## 2. Install the isolated pinned Linux/CUDA environment
Downloads several GB of dependencies. The notebook kernel is not the research interpreter.

In [ ]:
# Research packages stay separate from Colab's changing preinstalled environment.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.11.28'], check=True)
UV = [sys.executable, '-m', 'uv']
ENV = ROOT / 'research/.venv'
PY = ENV / 'bin/python'
if not PY.exists():
    subprocess.run(UV + ['venv', '--python', '3.12.13', str(ENV)], check=True)
UPSTREAM = ROOT / 'research/.cache/filbench-lighteval'
if not UPSTREAM.exists():
    UPSTREAM.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/filbench/lighteval.git', str(UPSTREAM)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', '7ad52795d7d218d4679f93a019b58dde4b749c5b'], check=True)
subprocess.run(UV + ['pip', 'sync', '--python', str(PY), '--torch-backend', 'cu126', 'requirements-colab.txt'],
               cwd=ROOT / 'research', check=True)
subprocess.run(UV + ['pip', 'check', '--python', str(PY)], check=True)


## 3. Run offline protocol tests; freeze the GPU execution profile

In [ ]:
subprocess.run([str(PY), '-m', 'unittest', 'discover', '-s', 'research/tests', '-v'], cwd=ROOT, check=True)
import shutil

def archive_results():
    # Includes manifests/logs/results, never model weights, caches, or environments.
    destination = '/content/makata-results-' + SESSION.name
    shutil.make_archive(destination, 'zip', root_dir=SESSION)
    return destination + '.zip'

def stage(command, candidate=0):
    log = SESSION / (command + '-' + str(candidate) + '.log')
    try:
        with log.open('w') as output:
            result = subprocess.run([str(PY), str(ROOT / 'research/colab/run.py'), command,
                                     '--session', str(SESSION), '--candidate', str(candidate)],
                                    cwd=ROOT, stdout=output, stderr=subprocess.STDOUT, timeout=14400)
        print(log.read_text()[-6000:])
        result.check_returncode()
    finally:
        print('Local runtime archive:', archive_results())

def download_results():
    from google.colab import files
    files.download(archive_results())

stage('freeze')


## 4. Verify Qwen and Sailor2 on development data only
Downloads each pinned checkpoint. Candidate runs execute in separate processes to release GPU memory.

In [ ]:
stage('smoke', 0)
stage('smoke', 1)
download_results()


## 5. Qwen: full 900-question baseline
Proceed only if both smoke checks succeeded. A four-hour timeout applies. No partial subset or CPU fallback is allowed.

In [ ]:
stage('evaluate', 0)
download_results()


## 6. Sailor2: full 900-question baseline
Keep the first downloaded archive before starting this run.

In [ ]:
stage('evaluate', 1)
download_results()


## 7. Confirm evidence and retain the final archive
A successful milestone requires two verified 900-question runs, not the synthetic pilot scores.

In [ ]:
for index in (0, 1):
    marker = json.loads((SESSION / ('baseline-' + str(index) + '.json')).read_text())
    directory = pathlib.Path(marker['directory'])
    saved = json.loads((directory / 'manifest.json').read_text())
    metrics = json.loads((directory / 'metrics.json').read_text())
    assert saved['status'] == 'complete' and saved['sample_count'] == 900
    print(saved['configuration']['model'], metrics['official_task_metrics'])
download_results()
print('Download finished? Then use Runtime → Disconnect and delete runtime to release the GPU.')
